<a href="https://www.kaggle.com/code/nguynvlun/chatbot-traffic?scriptVersionId=287392545" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Thư viện

In [1]:
# Đảm bảo protobuf đúng bản
!pip install -qq "protobuf==3.20.3"
!pip install docx2python python-docx
!pip install chromadb==0.5.15
!pip install -qq "transformers==4.44.2" \
               "accelerate==0.34.2" \
               "peft==0.12.0" \
               "triton==2.3.0"
!pip install -q \
    sentencepiece \
    tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.14.0
    Uninstalling typing_extensions-4.14.0:
      Successfully uninstalled typing_extensions-4.14.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
ibis-framework 9.5.0 requir

# Xử lý văn bản

In [2]:
import re, json, uuid, zipfile, unicodedata
from pathlib import Path
from tqdm import tqdm
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table

# =========================
# REGEX
# =========================
RX_CHUONG  = re.compile(r"^\s*Chương\s+[IVXLCDM]+\b", re.I)
RX_DIEU    = re.compile(r"^\s*Điều\s+\d+(\.\d+)?\b", re.I)
RX_KHOANMU = re.compile(r"^\s*(Khoản\s+\d+|\d+[\.\)])\s*", re.I)  # dùng làm ngữ cảnh bảng
RX_DOCTYPE = re.compile(r"^\s*(THÔNG TƯ|NGHỊ ĐỊNH|QUYẾT ĐỊNH|NGHỊ QUYẾT|LUẬT)\b", re.I)
RX_BANG    = re.compile(r"^\s*Bảng\s+\d+\b", re.I)
RX_CAN_CU  = re.compile(r"^\s*Căn\s*cứ\b", re.I)
RX_DASHLINE = re.compile(r"^\s*[-–—_\.|]+\s*$")

WS = re.compile(r"[ \t\u00A0]+")

# =========================
# UTILS
# =========================
def norm(s: str) -> str:
    s = (s or "").replace("\n", " ")
    return WS.sub(" ", s).strip()

def clean_text(s: str) -> str:
    s = norm(s)
    return re.sub(r"\n{3,}", "\n\n", s)

def is_docx_ok(fp: Path) -> bool:
    return zipfile.is_zipfile(fp)

def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))

def slugify_vi(s: str) -> str:
    s = strip_accents(norm(s)).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s or "col"

def try_parse_number(s: str):
    t = norm(s).replace(".", "").replace(",", ".")
    # giữ số âm, số thập phân
    if re.fullmatch(r"-?\d+(\.\d+)?", t):
        if "." in t:
            return float(t)
        return int(t)
    return None

# =========================
# ITER BLOCKS IN DOCX ORDER
# =========================
def iter_blocks_docx(doc: Document):
    """
    Yield ("p", text) or ("tbl", Table) in document order.
    """
    body = doc._element.body
    for child in body.iterchildren():
        tag = child.tag.lower()
        if tag.endswith("}p"):
            p = Paragraph(child, doc)
            t = norm(p.text)
            if t:
                yield ("p", t)
        elif tag.endswith("}tbl"):
            yield ("tbl", Table(child, doc))

# =========================
# DOC HEADER (DOCTYPE + TITLE)
# =========================
def extract_doc_type_and_title(stream):
    """
    stream: list[(kind, content)] where content is text for p, Table for tbl
    Return (doc_type, title, start_idx)
    """
    doc_type = ""
    title = ""
    start_idx = 0

    for i, (kind, content) in enumerate(stream):
        if kind != "p":
            continue
        line = norm(content)
        if not line:
            continue

        if RX_CHUONG.match(line):
            start_idx = i
            break
        if RX_CAN_CU.match(line):
            start_idx = i
            break

        if not doc_type and RX_DOCTYPE.match(line):
            doc_type = line
            continue

        if doc_type and not title:
            if RX_DASHLINE.match(line):
                continue
            if RX_CAN_CU.match(line):
                start_idx = i
                break
            title = line
            continue

    return clean_text(doc_type), clean_text(title), start_idx

# =========================
# TABLE -> UNIFIED SCHEMA
# =========================
def safe_row_cells(row):
    """
    Trả về list text của cell trong 1 row,
    không crash khi gặp vMerge/gridSpan lỗi của python-docx
    """
    out = []
    try:
        # cách chuẩn (nhanh)
        for c in row.cells:
            out.append(norm(c.text))
        return out
    except ValueError:
        # fallback: đọc trực tiếp XML <w:tc>
        out = []
        for tc in row._tr.tc_lst:
            # lấy text từ paragraph trong cell
            texts = []
            for p in tc.iterchildren():
                if p.tag.endswith("}p"):
                    texts.append("".join(t.text or "" for t in p.iter()))
            out.append(norm(" ".join(texts)))
        return out


def table_to_grid(table: Table):
    rows = []
    max_len = 0

    for r in table.rows:
        cells = safe_row_cells(r)
        rows.append(cells)
        max_len = max(max_len, len(cells))

    # pad để các row cùng số cột (rất quan trọng cho header multi-level)
    for r in rows:
        if len(r) < max_len:
            r.extend([""] * (max_len - len(r)))

    # bỏ row rỗng
    rows = [r for r in rows if any(x for x in r)]
    return rows


def detect_data_start(rows):
    """
    Heuristic nhận diện header_depth.
    Trả về index bắt đầu data (data_start).
    Works cho:
    - bảng speed (hàng data có số ở các cột cuối)
    - bảng training (cột 1 là số TT)
    """
    if len(rows) <= 2:
        return 1

    def row_numeric_count(r):
        cnt = 0
        for x in r:
            if try_parse_number(x) is not None:
                cnt += 1
        return cnt

    for i in range(len(rows)):
        r = rows[i]
        c0 = r[0] if len(r) > 0 else ""
        c1 = r[1] if len(r) > 1 else ""

        # Case bảng có số TT ở cột 1
        if c0.isdigit() and len(c1) >= 3:
            return i

        # Case bảng speed: data row thường có >= 1 số ở cột cuối
        if i >= 1 and row_numeric_count(r) >= 1:
            # tránh nhầm header có (km/h), (giờ)…
            headerish = any(("km/h" in x.lower() or "giờ" in x.lower() or "tốc độ" in x.lower()) for x in r)
            if not headerish:
                return i

    # fallback
    return 2

def build_header_paths(header_rows):
    """
    header_rows: rows[:data_start]
    Return: paths_per_col: list[list[str]]
    - mỗi cột là list các nhãn header từ trên xuống dưới
    """
    n_cols = max(len(r) for r in header_rows) if header_rows else 0
    paths = [[] for _ in range(n_cols)]

    for r in header_rows:
        for j in range(n_cols):
            cell = r[j] if j < len(r) else ""
            cell = norm(cell)
            if not cell:
                continue
            # tránh lặp y hệt liên tiếp
            if paths[j] and paths[j][-1].lower() == cell.lower():
                continue
            paths[j].append(cell)

    # loại bỏ path rỗng -> đặt default
    for j in range(n_cols):
        if not paths[j]:
            paths[j] = [f"Cột {j+1}"]
    return paths

def infer_value_columns(rows, data_start, n_cols):
    """
    Dùng tỷ lệ numeric theo cột trên data rows để đoán value columns.
    """
    data = rows[data_start:]
    if not data:
        return []

    ratios = []
    for j in range(n_cols):
        num = 0
        total = 0
        for r in data:
            if j >= len(r):
                continue
            total += 1
            if try_parse_number(r[j]) is not None:
                num += 1
        ratio = (num / total) if total else 0.0
        ratios.append(ratio)

    # cột value thường numeric cao
    value_cols = [j for j, rt in enumerate(ratios) if rt >= 0.5]
    # nếu không tìm thấy (bảng chữ), fallback: coi cột cuối là value
    if not value_cols and n_cols >= 2:
        value_cols = [n_cols - 1]
    return value_cols

def guess_unit_from_headers(paths_per_col):
    allh = " ".join(" ".join(p) for p in paths_per_col).lower()
    if "km/h" in allh:
        return "km/h"
    if "giờ" in allh:
        return "giờ"
    if "đồng" in allh or "vnđ" in allh:
        return "đồng"
    return None

def make_rendered_text(row_path, unit, col_members, cells):
    """
    Bullet text deterministic cho RAG.
    row_path: list[str] (các mức section/stt/nội dung…)
    """
    label = " – ".join([x for x in row_path if x])
    if unit:
        head = f"- {label} ({unit}):"
    else:
        head = f"- {label}:"
    lines = [head]
    for m in col_members:
        k = m["key"]
        v = cells.get(k, {}).get("value", "")
        if v == "" or v is None:
            continue
        sub_label = " – ".join(m.get("labels", []) or [m["path"][-1]])
        lines.append(f"  + {sub_label}: {v}")
    return "\n".join(lines)

def build_table_schema(table: Table, meta_ctx: dict):
    """
    meta_ctx chứa: source, doc_type, preamble, chuong, dieu, khoan_muc, table_id, table_title
    """
    rows = table_to_grid(table)
    if not rows or all(len(r) <= 1 for r in rows):
        return None  # bỏ bảng layout rác

    data_start = detect_data_start(rows)
    header_rows = rows[:data_start]
    n_cols = max(len(r) for r in rows)

    paths = build_header_paths(header_rows)
    value_cols = infer_value_columns(rows, data_start, n_cols)

    # row dimension cols = phần còn lại
    row_dim_cols = [j for j in range(n_cols) if j not in value_cols]
    # loại cột "SỐ TT" nếu muốn, vẫn giữ trong row_path
    # (giữ lại để bám đúng văn bản)

    unit = guess_unit_from_headers(paths)

    # build row_dimensions
    row_dimensions = []
    for j in row_dim_cols:
        row_dimensions.append({
            "name": paths[j][-1],
            "path": paths[j],
            "key": slugify_vi(paths[j][-1])
        })

    # build col_dimensions (1 trục chính)
    members = []
    for j in value_cols:
        p = paths[j]
        leaf = p[-1]
        key = slugify_vi(" ".join(p[-2:]) if len(p) >= 2 else leaf)
        members.append({
            "key": key,
            "path": p,
            "labels": p[1:] if len(p) > 1 else p
        })

    if value_cols and paths[value_cols[0]]:
        cd_name = paths[value_cols[0]][0]
    else:
        cd_name = "Giá trị"
    
    col_dimensions = [{
        "name": cd_name,
        "key": slugify_vi(cd_name),
        "unit": unit,
        "members": members
    }]


    # build data rows
    out_rows = []
    rendered_blocks = []
    for i, r in enumerate(rows[data_start:], start=data_start):
        # row_path
        row_path = []
        for j in row_dim_cols:
            row_path.append(r[j] if j < len(r) else "")

        # cells
        cell_map = {}
        for idx, j in enumerate(value_cols):
            mem = members[idx]  # đảm bảo đúng member của đúng cột value
            raw = r[j] if j < len(r) else ""
            val = try_parse_number(raw)
            if val is None:
                val = raw
            cell_map[mem["key"]] = {"value": val, "unit": unit}


        row_id = slugify_vi(" ".join([x for x in row_path if x]) or f"row_{i}")
        out_rows.append({
            "row_id": row_id,
            "row_path": [x for x in row_path if x],
            "cells": cell_map
        })

        rendered_blocks.append(make_rendered_text([x for x in row_path if x], unit, members, cell_map))

    schema = {
        "type": "table",
        "id": str(uuid.uuid4()),
        **meta_ctx,
        "table_kind": "matrix" if len(value_cols) >= 2 else "list",
        "row_dimensions": row_dimensions,
        "col_dimensions": col_dimensions,
        "rows": out_rows,
        "rendered_text": clean_text("\n".join(rendered_blocks)),
        "raw": {
            "header_depth": data_start,
            "n_rows": len(rows),
            "n_cols": n_cols,
            "value_cols": value_cols,
            "row_dim_cols": row_dim_cols
        }
    }
    return schema

# =========================
# PARSE DOCX -> article + table records
# =========================
def parse_docx(fp: Path):
    doc = Document(str(fp))
    stream = list(iter_blocks_docx(doc))
    doc_type, preamble_text, start_idx = extract_doc_type_and_title(stream)

    records = []
    cur = {"chuong": None, "dieu": None}

    # gom theo khoản
    cur_khoan = {"label": None, "text": []}

    last_khoan_muc = None
    last_table_id = None
    last_table_title = None

    def flush_khoan():
        if cur["dieu"] and cur_khoan["text"]:
            records.append({
                "type": "khoan",
                "id": str(uuid.uuid4()),
                "source": fp.name,
                "doc_type": doc_type,
                "preamble": preamble_text,
                "chuong": cur["chuong"],
                "dieu": cur["dieu"],
                "khoan_muc": cur_khoan["label"],   # <-- QUAN TRỌNG
                "text": clean_text("\n".join(cur_khoan["text"]))
            })
        cur_khoan["label"] = None
        cur_khoan["text"] = []

    def reset_on_new_dieu_or_chuong():
        flush_khoan()
        last_khoan_muc = None

    for kind, content in stream[start_idx:]:
        if kind == "p":
            t = norm(content)
            if not t:
                continue

            # Bảng X (id)
            if RX_BANG.match(t):
                last_table_id = t
                continue

            # Chương
            if RX_CHUONG.match(t):
                reset_on_new_dieu_or_chuong()
                cur["chuong"] = t
                cur["dieu"] = None
                last_table_id = None
                last_table_title = None
                continue

            # Điều
            if RX_DIEU.match(t):
                reset_on_new_dieu_or_chuong()
                cur["dieu"] = t
                last_table_id = None
                last_table_title = None
                continue

            # tiêu đề bảng heuristic
            if any(k in t.lower() for k in ["tốc độ", "khối lượng", "phân bổ", "mức", "thời gian", "khoảng cách"]):
                last_table_title = t

            # Khoản/Điểm/mục trong Điều
            mkm = RX_KHOANMU.match(t)
            if mkm and cur["dieu"]:
                # gặp khoản mới -> flush khoản cũ
                flush_khoan()
                last_khoan_muc = mkm.group(0).strip()
                cur_khoan["label"] = last_khoan_muc

            # nếu chưa có label khoản mà đang trong Điều: gom vào "mở đầu điều"
            if cur["dieu"] and cur_khoan["label"] is None:
                cur_khoan["label"] = "Mở đầu"

            if cur["dieu"]:
                cur_khoan["text"].append(t)

        else:  # tbl
            if not cur["dieu"]:
                continue

            meta_ctx = {
                "source": fp.name,
                "doc_type": doc_type,
                "preamble": preamble_text,
                "chuong": cur["chuong"],
                "dieu": cur["dieu"],
                "khoan_muc": last_khoan_muc,
                "table_id": last_table_id,
                "table_title": last_table_title or None
            }

            tbl_schema = build_table_schema(content, meta_ctx)
            if tbl_schema:
                records.append(tbl_schema)

                # KHÔNG trộn table vào text của khoản/điều (khuyến nghị)
                # cur_khoan["text"].append(tbl_schema["rendered_text"])

            last_table_id = None
            last_table_title = None

    flush_khoan()
    return records

# =========================
# RUN
# =========================
def run(in_path, out_path="data_struct/law.jsonl"):
    outp = Path(out_path)
    outp.parent.mkdir(parents=True, exist_ok=True)

    in_path = Path(in_path)
    if in_path.is_file():
        files = [in_path]
    elif in_path.is_dir():
        files = sorted(list(in_path.rglob("*.docx")) + list(in_path.rglob("*.DOCX")))
    else:
        raise FileNotFoundError(in_path)

    with open(outp, "w", encoding="utf-8") as f:
        for fp in tqdm(files):
            if not is_docx_ok(fp):
                print(f"[SKIP] Không phải DOCX hợp lệ: {fp}")
                continue
            recs = parse_docx(fp)
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print("Done ->", outp)

# Khi đã xác nhận thấy đúng file, chạy:
run("/kaggle/input/law-word", "data_struct/law.jsonl")


100%|██████████| 10/10 [00:01<00:00,  6.09it/s]

Done -> data_struct/law.jsonl


# Chunking

In [3]:
import json, re, uuid
from pathlib import Path
from tqdm import tqdm

# =============== CONFIG ===============
IN_PATH  = "data_struct/law.jsonl"
OUT_PATH = "data_struct/law_chunks.jsonl"

CHUNK_CHARS = 1200   # ~ 350-500 tokens tiếng Việt (ước lượng)
OVERLAP_CHARS = 150  # ~ 60-90 tokens

# =============== UTILS ===============
WS = re.compile(r"[ \t\u00A0]+")
def norm(s: str) -> str:
    s = (s or "").replace("\r", "")
    s = WS.sub(" ", s)
    return s.strip()

def split_blocks(text: str):
    """
    Tách theo khối ưu tiên:
    - dòng trống
    - bullet "- " / "+ "
    - các mục "1." "2." "Khoản ..."
    Giữ nguyên thứ tự.
    """
    text = text.replace("\r", "")
    lines = [ln.rstrip() for ln in text.split("\n")]

    blocks = []
    buf = []

    def flush():
        nonlocal buf
        if buf:
            blocks.append(norm("\n".join(buf)))
            buf = []

    for ln in lines:
        raw = ln.strip()
        if not raw:
            flush()
            continue

        is_new_block = (
            raw.startswith("- ")
            or raw.startswith("+ ")
            or re.match(r"^\d+[\.\)]\s+", raw) is not None
            or re.match(r"^Khoản\s+\d+\b", raw, re.I) is not None
            or re.match(r"^Điểm\s+[a-z]\b", raw, re.I) is not None
        )

        if is_new_block and buf:
            flush()

        buf.append(raw)

    flush()
    return [b for b in blocks if b]

def pack_blocks_to_chunks(blocks, chunk_chars, overlap_chars):
    """
    Gom blocks thành chunks, không vượt chunk_chars quá nhiều.
    Overlap theo ký tự ở ranh giới.
    """
    chunks = []
    cur = ""

    def push_chunk(s):
        s = norm(s)
        if s:
            chunks.append(s)

    for b in blocks:
        if not cur:
            cur = b
            continue

        if len(cur) + 2 + len(b) <= chunk_chars:
            cur = cur + "\n" + b
        else:
            push_chunk(cur)

            # tạo overlap: lấy tail của cur
            tail = cur[-overlap_chars:] if overlap_chars > 0 else ""
            cur = norm(tail + "\n" + b) if tail else b

    push_chunk(cur)
    return chunks

def chunk_text(text: str, chunk_chars=CHUNK_CHARS, overlap_chars=OVERLAP_CHARS):
    text = norm(text)
    if not text:
        return []
    blocks = split_blocks(text)
    if not blocks:
        return []
    return pack_blocks_to_chunks(blocks, chunk_chars, overlap_chars)

# =============== MAIN ===============
def run_chunking(in_path=IN_PATH, out_path=OUT_PATH):
    in_path = Path(in_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    n_in = 0
    n_out = 0

    with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
        for line in tqdm(fin, desc="Chunking"):
            line = line.strip()
            if not line:
                continue
            n_in += 1
            rec = json.loads(line)

            rtype = rec.get("type")
            parent_id = rec.get("id")

            # ưu tiên text nào để chunk
            if rtype == "table":
                text = rec.get("rendered_text") or ""
            else:
                text = rec.get("text") or ""

            chunks = chunk_text(text)
            for idx, ch in enumerate(chunks):
                out = {
                    "id": str(uuid.uuid4()),
                    "parent_id": parent_id,
                    "type": rtype,
                    "chunk_index": idx,
                    "text": ch,

                    # metadata quan trọng cho retrieval/citation
                    "source": rec.get("source"),
                    "doc_type": rec.get("doc_type"),
                    "preamble": rec.get("preamble"),
                    "chuong": rec.get("chuong"),
                    "dieu": rec.get("dieu"),
                    "khoan_muc": rec.get("khoan_muc"),

                    # table meta
                    "table_id": rec.get("table_id"),
                    "table_title": rec.get("table_title"),
                }
                fout.write(json.dumps(out, ensure_ascii=False) + "\n")
                n_out += 1

    print(f"Done. in_records={n_in}, out_chunks={n_out} -> {out_path}")

run_chunking("data_struct/law.jsonl", "data_struct/law_chunks.jsonl")

Chunking: 2409it [00:00, 6428.14it/s]

Done. in_records=2409, out_chunks=2347 -> data_struct/law_chunks.jsonl


# RAG

In [4]:
# RAG

import argparse, json, os, re, uuid
from pathlib import Path
import chromadb
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# =========================
# CONFIG
# =========================
EMB = os.getenv("RAG_EMB", "intfloat/multilingual-e5-small")
LLM = os.getenv("RAG_LLM", "Qwen/Qwen2.5-1.5B-Instruct")  # đổi sang đường dẫn local nếu bạn tải sẵn

DEFAULT_DB = "vectordb"
COL_NAME = "law_chunks"

# =========================
# EMBEDDING (E5)
# =========================
WS = re.compile(r"[ \t\u00A0]+")

def norm(s: str) -> str:
    return WS.sub(" ", (s or "").replace("\r", "").strip())

def mean_pool(last_hidden_state, attention_mask):
    m = attention_mask.unsqueeze(-1)
    v = (last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)
    return torch.nn.functional.normalize(v, p=2, dim=1)

class E5:
    def __init__(self, name=EMB):
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(name)
        self.mdl = AutoModel.from_pretrained(name).to(self.dev).eval()

    @torch.no_grad()
    def enc_passages(self, texts, max_len=512, batch=16):
        out=[]
        for i in range(0, len(texts), batch):
            chunk = [f"passage: {t}" for t in texts[i:i+batch]]
            toks = self.tok(
                chunk, padding=True, truncation=True,
                max_length=max_len, return_tensors="pt"
            ).to(self.dev)
            v = mean_pool(self.mdl(**toks).last_hidden_state, toks["attention_mask"])
            out.append(v.cpu())
        return torch.cat(out, 0).numpy()

    @torch.no_grad()
    def enc_query(self, q, max_len=512):
        toks = self.tok(
            [f"query: {q}"], padding=True, truncation=True,
            max_length=max_len, return_tensors="pt"
        ).to(self.dev)
        v = mean_pool(self.mdl(**toks).last_hidden_state, toks["attention_mask"])
        return v.cpu().numpy()[0].tolist()

def clean_meta(d: dict):
    out = {}
    for k, v in (d or {}).items():
        if v is None:
            continue
        if isinstance(v, (bool, int, float, str)):
            out[k] = v
        else:
            out[k] = str(v)
    return out

# =========================
# SIMPLE RERANK (keyword/coverage)
# =========================
RX_TOKEN = re.compile(r"[A-Za-zÀ-ỹ0-9]+", re.U)

STOP = {
    "tốc", "độ", "tối", "đa", "bao", "nhiêu", "là", "gì", "trong", "ở", "theo", "khi",
    "và", "hoặc", "cho", "của", "các", "một", "những", "đến", "với", "trên", "dưới",
}

def keywords(q: str):
    toks = [t.lower() for t in RX_TOKEN.findall(q)]
    toks = [t for t in toks if len(t) >= 2 and t not in STOP]
    # giữ top unique theo thứ tự
    seen = set()
    out = []
    for t in toks:
        if t not in seen:
            out.append(t); seen.add(t)
    return out[:12]

def score_doc(qk, text, meta):
    t = (text or "").lower()
    m = " ".join([str(v) for v in (meta or {}).values()]).lower()

    hit = sum(1 for k in qk if (k in t or k in m))
    bonus = 0

    # bonus theo domain luật giao thông
    if "điều" in m: bonus += 1
    if meta.get("table_id"): bonus += 1
    if "bảng" in t: bonus += 0.5
    if "km/h" in t: bonus += 0.5

    # phạt nhẹ docs quá ngắn
    length_pen = 0.5 if len(text) < 200 else 0

    return hit + bonus - length_pen

def rerank(question, docs, metas, keep=6):
    qk = keywords(question)
    scored = []
    for d, m in zip(docs, metas):
        scored.append((score_doc(qk, d, m), d, m))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:keep]
    return [x[1] for x in top], [x[2] for x in top], top

# =========================
# QUERY REWRITE (multi-query)
# =========================
def rewrite_question_rule(q: str, n: int = 6):
    q0 = (q or "").strip()
    ql = q0.lower()

    variants = []
    seen = set()

    def add(s):
        s = (s or "").strip()
        if not s:
            return
        k = s.lower()
        if k in seen:
            return
        seen.add(k)
        variants.append(s)

    add(q0)

    # thêm từ khóa truy xuất luật
    add(q0 + " mức phạt theo quy định hiện hành")
    add(q0 + " căn cứ Điều, Khoản/Bảng nào?")
    add("Theo Nghị định 168/2024/NĐ-CP, " + q0)

    # chuẩn hóa thuật ngữ phương tiện
    if "xe máy" in ql:
        add(q0.replace("xe máy", "xe mô tô, xe gắn máy"))
        add("Mức phạt đối với người điều khiển xe mô tô, xe gắn máy khi " + q0)

    # hướng bảng tốc độ
    if "tốc độ" in ql or "vượt" in ql:
        add("Bảng mức phạt vượt tốc độ đối với xe mô tô, xe gắn máy: " + q0)
        add(q0 + " vượt quá bao nhiêu km/h thì bị phạt?")

    # tách intent phụ hay gặp
    add(q0 + " có bị trừ điểm giấy phép lái xe không?")
    add(q0 + " có bị tước quyền sử dụng giấy phép lái xe không?")

    return variants[:n]


def merge_results(all_docs, all_metas, max_items=60):
    """
    Gộp kết quả từ nhiều query. Unique theo (parent_id, chunk_index) nếu có.
    """
    docs2, metas2 = [], []
    seen = set()

    for d, m in zip(all_docs, all_metas):
        pid = (m or {}).get("parent_id") or ""
        cix = str((m or {}).get("chunk_index") or "")
        key = f"{pid}::{cix}"
        if key == "::":  # không có meta, fallback theo text
            key = (d or "")[:160]

        if key in seen:
            continue
        seen.add(key)

        docs2.append(d)
        metas2.append(m)

        if len(docs2) >= max_items:
            break

    return docs2, metas2


# =========================
# LLM (Qwen)
# =========================
def load_llm(model_name=LLM):
    tok = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True,
        trust_remote_code=True
    )

    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,   # chạy GPU ổn định
        device_map="auto",           # 3B fit gọn 1 GPU
        trust_remote_code=True,
    ).eval()

    return tok, mdl


def build_context(docs, metas, max_chars=9000, preamble_max=260):
    blocks = []
    total = 0

    for i, (d, m) in enumerate(zip(docs, metas), 1):
        source = (m.get("source") or "").strip()
        doc_type = (m.get("doc_type") or "").strip()
        chuong = (m.get("chuong") or "").strip()
        dieu = (m.get("dieu") or "").strip()
        khoan = (m.get("khoan_muc") or "").strip()
        table_id = (m.get("table_id") or "").strip()

        preamble = (m.get("preamble") or "").strip()
        # cắt ngắn preamble để không làm nổ context (tùy bạn)
        if preamble and len(preamble) > preamble_max:
            preamble = preamble[:preamble_max].rstrip() + "…"

        ref_parts = [x for x in [source, doc_type, chuong, dieu, khoan, table_id] if x]
        ref = " | ".join(ref_parts)

        # nhấn mạnh Điều + preamble trong context block
        head_lines = [f"[{i}] {ref}".strip()]
        if dieu:
            head_lines.append(f"Điều: {dieu}")
        if preamble:
            head_lines.append(f"Preamble: {preamble}")

        block = "\n".join(head_lines) + "\n" + (d or "").strip()

        if total + len(block) + 2 > max_chars:
            break
        blocks.append(block)
        total += len(block) + 2

    return "\n\n".join(blocks)


SYSTEM_PROMPT = (
    "Bạn là trợ lý pháp luật giao thông Việt Nam.\n"
    "CHỈ trả lời dựa trên CONTEXT.\n"
    "Nếu CONTEXT không đủ để kết luận, hãy nói rõ: "
    "'Không đủ căn cứ trong CONTEXT để trả lời chính xác.' "
    "và hỏi lại đúng 1 câu để lấy thông tin còn thiếu.\n"
    "Khi trả lời, bắt buộc có:\n"
    "1) Kết luận ngắn gọn\n"
    "2) Căn cứ: liệt kê theo từng nguồn [số], và MỖI nguồn phải kèm:\n"
    "   - Điều (ví dụ: Điều 5)\n"
    "   - Preamble (trích hoặc tóm tắt ngắn phần preamble nếu có)\n"
    "   - Nếu là bảng thì ghi thêm Bảng/table_id\n"
)


def generate_answer(tok, mdl, question, context, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nCÂU HỎI:\n{question}\n\nTRẢ LỜI:"}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=4096).to(mdl.device)

    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
        )
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip()

# =========================
# CHROMA INDEX/QUERY
# =========================
def get_collection(db_path):
    client = chromadb.PersistentClient(path=db_path)
    return client.get_or_create_collection(COL_NAME)

def cmd_index(args):
    enc = E5()
    col = get_collection(args.db)

    ids, docs, metas = [], [], []
    with open(args.in_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            r = json.loads(line)
            txt = (r.get("text") or "").strip()
            if not txt:
                continue

            doc_id = str(r.get("id") or f"chunk_{idx}")
            ids.append(doc_id)
            docs.append(txt)

            meta = {
                "source": r.get("source") or "",
                "doc_type": r.get("doc_type") or "",
                "preamble": r.get("preamble") or "",
                "chuong": r.get("chuong") or "",
                "dieu": r.get("dieu") or "",
                "khoan_muc": r.get("khoan_muc") or "",
                "type": r.get("type") or "",
                "parent_id": r.get("parent_id") or "",
                "chunk_index": int(r.get("chunk_index") or 0),
                "table_id": r.get("table_id") or "",
                "table_title": r.get("table_title") or "",
            }
            metas.append(clean_meta(meta))

    if not docs:
        print("No docs to index."); return

    emb = enc.enc_passages(docs, batch=args.batch).tolist()
    B = 1024
    for i in range(0, len(ids), B):
        col.add(
            ids=ids[i:i+B],
            documents=docs[i:i+B],
            embeddings=emb[i:i+B],
            metadatas=metas[i:i+B],
        )
    print(f"Indexed {len(ids)} chunks -> {args.db}/{COL_NAME}")

def cmd_ask(args):
    enc = E5()
    col = get_collection(args.db)

    # 1) rewrite -> tạo nhiều câu hỏi truy xuất
    queries = rewrite_question_rule(args.q, n=getattr(args, "n_rewrite", 6))

    # 2) query nhiều lần -> gộp results
    all_docs, all_metas = [], []
    for q in queries:
        qvec = enc.enc_query(q)
        res = col.query(query_embeddings=[qvec], n_results=args.topk)
        docs = res.get("documents", [[]])[0]
        metas = res.get("metadatas", [[]])[0]
        all_docs.extend(docs)
        all_metas.extend(metas)

    docs, metas = merge_results(all_docs, all_metas, max_items=getattr(args, "merge_max", 60))

    if not docs:
        print("Không tìm thấy căn cứ phù hợp."); return

    # 3) rerank theo câu hỏi gốc
    docs2, metas2, scored = rerank(args.q, docs, metas, keep=args.k)

    # 4) build context + answer
    context = build_context(docs2, metas2, max_chars=args.max_ctx_chars)
    tok, mdl = load_llm(args.llm)
    ans = generate_answer(tok, mdl, args.q, context, max_new_tokens=args.max_new_tokens)

    if args.show_hits:
        print("=== REWRITE QUERIES ===")
        for qq in queries:
            print("-", qq)
        print()

        print("=== TOP (after rerank) ===")
        for i, (sc, d, m) in enumerate(scored[:args.k], 1):
            ref = " | ".join([x for x in [m.get("source"), m.get("dieu"), m.get("khoan_muc"), m.get("table_id")] if x])
            print(f"[{i}] score={sc:.2f} :: {ref}")
        print()

    print(ans)

cmd_index(type("NS", (), {
    "in_path": "data_struct/law_chunks.jsonl",
    "db": "vectordb",
    "batch": 16
})())
cmd_ask(type("NS", (), {
    "db": "vectordb",
    "q": "khoảng cách giữa hai xe là bao nhiêu",
    "topk": 20,
    "k": 10,
    "llm": LLM,
    "max_ctx_chars": 9000,
    "max_new_tokens": 256,
    "show_hits": True,
    "n_rewrite": 6,     # số câu rewrite
    "merge_max": 60    # số docs gộp tối đa trước rerank
})())

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


=== REWRITE QUERIES ===
- khoảng cách giữa hai xe là bao nhiêu
- khoảng cách giữa hai xe là bao nhiêu mức phạt theo quy định hiện hành
- khoảng cách giữa hai xe là bao nhiêu căn cứ Điều, Khoản/Bảng nào?
- Theo Nghị định 168/2024/NĐ-CP, khoảng cách giữa hai xe là bao nhiêu
- khoảng cách giữa hai xe là bao nhiêu có bị trừ điểm giấy phép lái xe không?
- khoảng cách giữa hai xe là bao nhiêu có bị tước quyền sử dụng giấy phép lái xe không?

=== TOP (after rerank) ===
[1] score=7.50 :: 38_2024_TT-BGTVT_622477.docx | Điều 11. Khoảng cách an toàn giữa hai xe khi tham gia giao thông trên đường bộ | 2. | Bảng 3
[2] score=6.50 :: 38_2024_TT-BGTVT_622477.docx | Điều 11. Khoảng cách an toàn giữa hai xe khi tham gia giao thông trên đường bộ | 2.
[3] score=6.50 :: 38_2024_TT-BGTVT_622477.docx | Điều 6. Tốc độ khai thác tối đa cho phép xe cơ giới tham gia giao thông trên đường bộ (trừ đường cao tốc) | 1. | Bảng 1
[4] score=6.50 :: 38_2024_TT-BGTVT_622477.docx | Điều 6. Tốc độ khai thác tối đa cho phép